# VQE: build a circuit, calculate its energy
### Student notebook

Use gate matrices to build a state, calculate its energy, then let SciPy adjust the gate angles. Here, **ansatz** simply means our chosen circuit with adjustable angles.

Run the cells in order and complete the `TODO`s. The simulator, optimisation and plotting code are provided; you mainly write circuits and calculate `psi.conj().T @ H @ psi`. **Run All will stop at unfinished exercises.**

We use $J=1$, angles in radians, and ideal simulated states: no measurement sampling or hardware noise. Two qubits have **one** $ZZ$ bond; the six-qubit ring has **six**, including $(5,0)$.

In [ ]:
# Uncomment if these packages are missing, then restart the kernel.
# %pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from collections import namedtuple

np.set_printoptions(precision=5, suppress=True)

## 1. Bring your own quantum simulator

A program is a list of `Instruction(gate_matrix, qubit_indices)` entries. Gate functions such as `RY(0.5)` return ordinary NumPy matrices; there are no symbolic parameters.

The supplied simulator follows the original tutorial: build the initial state, apply each gate and restore the qubit order. Qubit 0 is the leftmost bit in $|q_0q_1\cdots\rangle$. For a controlled gate, write `[control, target]`. Use `.ravel()` to turn the returned array into a flat vector before multiplying by a Hamiltonian.

In [ ]:
Instruction = namedtuple("Instruction", "gate bits")
ket0, ket1 = np.array([1, 0], complex), np.array([0, 1], complex)

def variational_quantum_simulator(program, nq, initial_qubits=None):
    if initial_qubits is None:
        initial_qubits = [ket0] * nq
    state = initial_qubits[0]
    for qubit in initial_qubits[1:]:
        state = np.kron(state, qubit)
    state = state.reshape([2] * nq)

    for instruction in program:
        bits = list(instruction.bits)
        k = len(bits)
        u = np.asarray(instruction.gate).reshape([2] * (2 * k))
        # Matrix columns are the inputs; matrix rows are the outputs.
        state = np.tensordot(state, u, axes=(bits, list(range(k, 2 * k))))
        current_order = [q for q in range(nq) if q not in bits] + bits
        state = state.transpose([current_order.index(q) for q in range(nq)])
    assert np.isclose(np.linalg.norm(state), 1), "Check your gates and initial state."
    return state

### Exercise 1 — gates as numerical functions

Complete the Pauli, Hadamard and CNOT matrices. Call the Hadamard `Had`, so that `H` can mean the Hamiltonian later. Then implement
$$R_P(\theta)=\cos(\theta/2)I-i\sin(\theta/2)P,$$
$$C(U)=|0\rangle\langle0|\otimes I+|1\rangle\langle1|\otimes U.$$
`np.kron(A, B)` forms $A\otimes B$; `np.outer(ket0, ket0)` forms $|0\rangle\langle0|$.

In [ ]:
I = np.eye(2, dtype=complex)
X = None    # TODO
Y = None    # TODO
Z = None    # TODO
Had = None  # TODO
CN = None   # TODO: first qubit controls the second

def rotation(P, theta):
    # TODO: return the matrix R_P(theta).
    raise NotImplementedError("Complete rotation")

def RX(theta): return rotation(X, theta)
def RY(theta): return rotation(Y, theta)
def RZ(theta): return rotation(Z, theta)

def controlled(U):
    # TODO: use np.outer and np.kron.
    raise NotImplementedError("Complete controlled")

def CRX(theta): return controlled(RX(theta))
def CRY(theta): return controlled(RY(theta))
def CRZ(theta): return controlled(RZ(theta))

In [ ]:
assert all(g is not None for g in [X, Y, Z, Had, CN]), "Complete the gate matrices."
for gate in [X, Y, Z, Had, CN, RX(0.4), RY(0.4), RZ(0.4), CRY(0.4)]:
    assert np.allclose(gate.conj().T @ gate, np.eye(len(gate)))
assert np.allclose(RY(0.4) @ ket0, [np.cos(0.2), np.sin(0.2)])
assert np.allclose(CRY(0.4) @ np.kron(ket0, ket0), np.kron(ket0, ket0))
assert np.allclose(CRY(0.4) @ np.kron(ket1, ket0), np.kron(ket1, RY(0.4) @ ket0))
print("Gate checks passed.")

### Exercise 2 — write a short program

Predict the output of the program below before running the cell. The initial state is always $\ket{0}^{\otimes{N}}$, so before the program $\ket{\psi_0}=\ket{00000}$ then $\ket{\psi}=U_{\rm{program}}\ket{\psi_0}$. 

In [ ]:
program1 = [Instruction(X, [0]), Instruction(X, [3])]
program2 = program1 + [Instruction(CN, [3, 0])]
for program in [program1, program2]:
    state = variational_quantum_simulator(program, 5)
    print("Occupied basis index:", np.argwhere(np.abs(state) > 0.99)[0])



Then make $(|00\rangle+|11\rangle)/\sqrt2$ using one Hadamard and one CNOT.

For example, `Instruction(RY(0.5), [2])` rotates qubit 2, and `Instruction(CRY(0.7), [3, 0])` has control 3 and target 0.

In [ ]:
bell_program = []  # TODO: add two instructions.
psi = variational_quantum_simulator(bell_program, 2).ravel()
assert np.allclose(psi, np.array([1, 0, 0, 1]) / np.sqrt(2))
print("Bell state:", psi)

## 2. First VQE: two qubits

Use the lecture convention, with just one interaction between the two qubits:
$$H_2(h)=-J\,Z\otimes Z-h(X\otimes I+I\otimes X).$$
For a normalised state, its energy is $E=\bra{\psi} H_2\ket{\psi}$. The code below builds the full $4\times4$ matrix.

In [ ]:
def H2(h, J=1.0):
    return -J * np.kron(Z, Z) - h * (np.kron(X, I) + np.kron(I, X))

print(H2(0.5).real)

### Two analytical checks

**The shared $R_y$ circuit.** Applying $R_y(\theta)$ to each qubit gives
$$|\psi(\theta)\rangle=
\left[\cos(\theta/2)|0\rangle+\sin(\theta/2)|1\rangle\right]^{\otimes2}.$$
Using $\langle Z\rangle=\cos\theta$ and $\langle X\rangle=\sin\theta$,
$$E_{\rm RY}(\theta)=-J\cos^2\theta-2h\sin\theta
=-J+J\left(\sin\theta-\frac hJ\right)^2-\frac{h^2}{J}.$$
So the best shared angle has $\sin\theta_\star=\min(h/J,1)$ for $h\geq0$. This is our two-qubit mean-field result, not the exact energy in general.

**The exact answer.** Let $|a\rangle=(|00\rangle+|11\rangle)/\sqrt2$ and $|b\rangle=(|01\rangle+|10\rangle)/\sqrt2$. Acting with $H_2$ gives
$$H_2|a\rangle=-J|a\rangle-2h|b\rangle,\qquad
H_2|b\rangle=-2h|a\rangle+J|b\rangle.$$
Thus we solve a $2\times2$ problem:
$$\det\begin{pmatrix}-J-E&-2h\\-2h&J-E\end{pmatrix}=0
\quad\Rightarrow\quad E^2=J^2+4h^2.$$
The two remaining, minus-sign combinations have energies $-J$ and $+J$. Therefore
$$\boxed{E_0(h)=-\sqrt{J^2+4h^2}.}$$
We will compare circuit energies with this answer; we do not use its eigenstate to prepare our circuit.

### Exercise 3 — define two circuits; the optimisation is provided

**Circuit A:** start in $|0\cdots0\rangle$ and apply `RY(thetas[0])` to **every qubit**, using the same angle. Write it for any number `nq` so that we can reuse it on six qubits. Return a list of instructions (a program), not a state.

**Circuit B (two qubits):** do Circuit A, then `CRY(thetas[1])` with **control 0, target 1**. There are now two adjustable angles. Return a list of instructions (a program), not a state.

In [ ]:
def ry_ansatz(thetas, nq):
    # TODO: RY(thetas[0]) on qubits 0, 1, ..., nq-1.
    raise NotImplementedError("Complete Circuit A") # you can remove this
    return program

def ry_cry_two(thetas, nq=2):
    program = ry_ansatz(thetas, nq)
    # TODO: append CRY(thetas[1]) with control 0 and target 1.
    raise NotImplementedError("Complete Circuit B") # you can remove this
    return program

### Provided: the VQE loop

For each set of angles: **build the circuit → get $\psi$ → compute its energy → let SciPy change the angles**. We try three starting choices and keep the lowest result found. This does not guarantee the absolute minimum.

Call `optimise(ansatz, n_angles, H, nq)`. It returns `theta_opt, psi_opt`. You only need to supply your circuit, the number of adjustable angles, the Hamiltonian matrix and the number of qubits. No optimiser settings need to be changed. [SciPy reference](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html).

In [ ]:
def optimise(ansatz, n_angles, H, nq):
    def energy(thetas):
        psi = variational_quantum_simulator(ansatz(thetas, nq), nq).ravel()
        return float((psi.conj().T @ H @ psi).real)

    rng = np.random.default_rng(7)
    best = None
    for repeat in range(3):
        start = rng.uniform(-np.pi, np.pi, n_angles)
        result = minimize(energy, start, method="BFGS")
        if best is None or result.fun < best.fun:
            best = result
    theta_opt = best.x
    psi_opt = variational_quantum_simulator(ansatz(theta_opt, nq), nq).ravel()
    return theta_opt, psi_opt

H = H2(h=1.0)
theta_opt, psi_opt = optimise(ry_cry_two, 2, H, 2)
E = (psi_opt.conj().T @ H @ psi_opt).real
print("Angles:", theta_opt)
print("Circuit energy:", E, " | Exact energy:", -np.sqrt(5))

### Provided: compare the two circuits over different fields

All energies are divided by the number of qubits. The same optimiser is used for both circuits. At $h=1$, does adding the controlled rotation reach the exact energy, or only improve it?

In [ ]:
h_values = np.linspace(0, 3, 10)
energies_ry, energies_cry = [], []
for h in h_values:
    H = H2(h)
    _, psi_ry = optimise(ry_ansatz, 1, H, 2)
    _, psi_cry = optimise(ry_cry_two, 2, H, 2)
    energies_ry.append((psi_ry.conj().T @ H @ psi_ry).real / 2)
    energies_cry.append((psi_cry.conj().T @ H @ psi_cry).real / 2)

plt.figure(figsize=(6.5, 4))
plt.plot(h_values, -np.sqrt(1 + 4*h_values**2) / 2, label="Exact")
plt.plot(h_values, energies_ry, "o--", label="RY on both qubits")
plt.plot(h_values, energies_cry, "s--", label="RY on both + CRY(0,1)")
plt.xlabel(r"$h/J$")
plt.ylabel(r"$E/(2J)$")
plt.title("Two qubits: one ZZ bond")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

**Your observation:** _Write one sentence._

## 3. Six qubits on a ring
### Exercise 4 — build the matrix and calculate one energy

Now use
$$\begin{aligned}H_6(h)&=-J(Z_0Z_1+Z_1Z_2+Z_2Z_3+Z_3Z_4+Z_4Z_5+Z_5Z_0)\\&\quad-h(X_0+X_1+X_2+X_3+X_4+X_5).\end{aligned}$$
Build its **$64\times64$ matrix**. The supplied `kron_all` just takes repeated Kronecker products. For example,
`kron_all([Z, Z, I, I, I, I])` is $Z_0Z_1$, and
`kron_all([X, I, I, I, I, I])` is $X_0$.

Complete the six $ZZ$ terms and the six $X$ terms in `H6(h)`. The argument `h` must change the field term; do not fix the field inside the function.

In [ ]:
def kron_all(operators):
    result = operators[0]
    for operator in operators[1:]:
        result = np.kron(result, operator)
    return result

def H6(h, J=1.0):
    zz = None  # TODO: sum the six ZZ matrices, including the bond (5,0).
    xx = None  # TODO: sum the six X matrices.
    return -J * zz - h * xx

In [ ]:
H = H6(0.4)
assert H.shape == (64, 64) and np.allclose(H, H.conj().T)
all_zero = kron_all([ket0] * 6)
all_plus = kron_all([(ket0 + ket1) / np.sqrt(2)] * 6)
assert np.isclose((all_zero.conj().T @ H @ all_zero).real, -6)
assert np.isclose((all_plus.conj().T @ H @ all_plus).real, -6 * 0.4)
print("Hamiltonian checks passed.")

**Now calculate one energy, without optimisation.** Use `ry_ansatz([0.5], 6)` to put $R_y(0.5)$ on every qubit. Obtain the flat vector `psi`, then calculate its energy at $h=1$ using `psi.conj().T @ H @ psi`. Take `.real` to remove numerical imaginary round-off.

In [ ]:
h = 1.0
H = H6(h)
psi = None  # TODO: simulate ry_ansatz([0.5], 6), then use .ravel().
E = None    # TODO: calculate the energy using psi and H.
assert psi is not None and E is not None, "Complete the state and energy."
print("Energy:", E, " | Energy per qubit:", E / 6)

**Connection to the lecture.** There is now one bond per qubit, so
$$\frac{E_{\rm RY}(\theta)}6=-\cos^2\theta-h\sin\theta
=\left(\sin\theta-\frac h2\right)^2-1-\frac{h^2}4.$$
Thus the best shared angle obeys $\sin\theta_\star=\min(h/2,1)$, and
$$\frac{E_{\rm MF}}6=\begin{cases}-1-h^2/4,&0\leq h\leq2,\\-h,&h\geq2.\end{cases}$$
This differs from the two-qubit formula because that example had only one bond in total. For our six-qubit matrix, `np.linalg.eigh(H)` gives exact energies and states to use as a check.

### Exercise 5 — add controlled rotations

Start with `RY(thetas[0])` on every qubit, as before. Then apply `CRX(thetas[1])` in this order, using the same second angle for every controlled gate:
$$0\to1,\quad1\to2,\quad2\to3,\quad3\to4,\quad4\to5,\quad5\to0.$$
The arrow points from **control to target**. In a general `nq`-qubit circuit, the final connection is `nq-1` to `0`.

In [ ]:
def ry_crx_ring(thetas, nq):
    program = ry_ansatz(thetas, nq)
    # TODO: CRX(thetas[1]) on (0,1), (1,2), ..., (nq-2,nq-1).
    # TODO: one final CRX(thetas[1]) on (nq-1,0).
    raise NotImplementedError("Complete the controlled rotations")
    return program

### Provided: compare energies and correlations

The cell below optimises both circuits at $h=1$. Each column returned by `eigh` is an eigenstate; the first has the lowest energy.

In [ ]:
h = 1.0
H = H6(h)
theta_ry, psi_ry6 = optimise(ry_ansatz, 1, H, 6)
theta_crx, psi_crx6 = optimise(ry_crx_ring, 2, H, 6)
exact_energies, exact_states = np.linalg.eigh(H)
psi_exact6 = exact_states[:, 0]

for name, state in [("RY", psi_ry6), ("RY + CRX", psi_crx6), ("Exact", psi_exact6)]:
    E = (state.conj().T @ H @ state).real
    print(f"{name:10s} E/6 = {E/6:.8f}")

To check whether spins at different positions line up, plot
$$G(r)=\langle Z_0Z_r\rangle=\psi^\dagger(Z_0Z_r)\psi.$$
On our six-qubit ring, $r=3$ is the largest separation. We plot $G(r)$ itself, **without subtracting** $\langle Z_0\rangle\langle Z_r\rangle$. A nonzero value can already occur for independent spins; this is not a measurement of a correlation length.

In [ ]:
def z_correlations(psi, nq=6):
    values = []
    for r in range(1, nq // 2 + 1):
        operators = [I] * nq
        operators[0] = operators[r] = Z
        ZZ = kron_all(operators)
        values.append((psi.conj().T @ ZZ @ psi).real)
    return values

r_values = [1, 2, 3]
plt.figure(figsize=(6.5, 4))
for name, state in [("RY", psi_ry6), ("RY + CRX", psi_cry6), ("Exact", psi_exact6)]:
    plt.plot(r_values, z_correlations(state), "o-", label=name)
plt.xticks(r_values)
plt.xlabel("Separation r")
plt.ylabel(r"$\langle Z_0 Z_r\rangle$")
plt.title("Six qubits, h/J = 1")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

**Explain:** Why is the RY-only curve flat? Does a better energy necessarily give a better value for every distant-spin correlation?

_Your answer:_

## 4. Your circuit: how low can you get the energy?
### Exercise 6 — change the circuit, not the optimisation code

Define `my_ansatz(thetas, nq)`. You could add another $R_y$ gate on each qubit, use different angles on different qubits, or repeat some gates. Keep starting from $|0\cdots0\rangle$; do not insert the exact eigenstate.

`n_angles` is the number of entries your circuit uses in `theta`. The starter below is the previous two-angle circuit. Change it, then run the supplied ten-field loop. **Complete just the energy calculation inside that loop**, using `psi_opt` and the current `H`, and store the energy per qubit in the provided list.

In [ ]:
def my_ansatz(thetas, nq):
    return ry_crx_ring(thetas, nq)  # Starting point: change or add gates.

n_angles = 2  # Update this when you introduce new angles.

In [ ]:
h_values = np.linspace(0, 3, 10)
my_energies, exact_curve = [], []
for h in h_values:
    H = H6(h)
    theta_opt, psi_opt = optimise(my_ansatz, n_angles, H, 6)
    E = None  # TODO: calculate (psi_opt dagger) H psi_opt, as a real number.
    assert E is not None, "Calculate E from psi_opt and H."
    my_energies.append(E / 6)
    exact_curve.append(np.linalg.eigvalsh(H)[0] / 6)

mf_curve = np.where(h_values <= 2, -1 - h_values**2 / 4, -h_values)
plt.figure(figsize=(6.5, 4))
plt.plot(h_values, exact_curve, label="Exact: six-qubit matrix")
plt.plot(h_values, mf_curve, "--", label="RY only: analytical minimum")
plt.plot(h_values, my_energies, "o--", label="My circuit")
plt.xlabel(r"$h/J$")
plt.ylabel(r"$E/(6J)$")
plt.title("Six qubits: compare energies")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

**Keep:** your circuit, its number of angles and the energy plot. Which change helped? Your energies should not fall below the exact curve, apart from numerical round-off. A poor result can also come from the angle search, not just the circuit.

## 5. The same idea in Qiskit

Only circuit construction and state simulation change. We keep the same Hamiltonian matrices, energy expression and optimisation recipe. No account or quantum device is needed for this local simulation.

Useful calls: `qc.ry(angle, qubit)`, `qc.cry(angle, control, target)`, and `qc.cx(control, target)`. `Statevector.from_instruction(qc)` simulates a circuit **without measurement instructions**. The provided `q_state` reorders its components to match our NumPy matrices; leave that helper unchanged. [Qiskit gates](https://quantum.cloud.ibm.com/docs/api/qiskit/qiskit.circuit.QuantumCircuit), [Statevector](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.Statevector), [qubit ordering](https://quantum.cloud.ibm.com/docs/en/guides/bit-ordering).

In [ ]:
# Uncomment if needed, then restart the kernel and run from the top.
# %pip install "qiskit>=2,<3"
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

def q_state(qc):
    return Statevector.from_instruction(qc).reverse_qargs().data

# Worked two-qubit example: the same RY + CRY circuit.
qc = QuantumCircuit(2)
qc.ry(0.5, 0)
qc.ry(0.5, 1)
qc.cry(0.7, 0, 1)
psi = q_state(qc)
H = H2(h=1.0)
# Check that the two simulators agree on the same gates.
assert np.allclose(psi, variational_quantum_simulator(ry_cry_two([0.5, 0.7], 2), 2).ravel())
print(qc.draw(output="text"))
print("Energy:", (psi.conj().T @ H @ psi).real)

### Provided: RY on every qubit

This function returns a Qiskit circuit. The example again evaluates RY(0.5) on six qubits at $h=1$. It should reproduce your earlier fixed-angle energy.

In [ ]:
def q_ry(thetas, nq):
    qc = QuantumCircuit(nq)
    for q in range(nq):
        qc.ry(thetas[0], q)
    return qc

H = H6(h=1.0)
psi = q_state(q_ry([0.5], 6))
print("Energy:", (psi.conj().T @ H @ psi).real)

### Exercise 7 — improve the Qiskit circuit

Start from `q_ry`. Add `qc.cry(thetas[1], control, target)` for **(0,1), (1,2), (2,3), (3,4), (4,5), (5,0)**, in that order. Then try the changes you made in your own NumPy circuit. Update `q_n_angles` to match the angles you use.

In [ ]:
def q_my_ansatz(thetas, nq):
    qc = q_ry(thetas, nq)
    # TODO: add the six controlled rotations, including (nq-1,0).
    # Then try improving the circuit further.
    raise NotImplementedError("Add your Qiskit gates")
    return qc

q_n_angles = 2

### Provided: optimise and compare

The helper below is the same recipe as before, with `q_state` replacing our simulator. The ten-field loop, Hamiltonians and energy plot are supplied. You only change the circuit above.

In [ ]:
def q_optimise(ansatz, n_angles, H, nq):
    def energy(theta):
        psi = q_state(ansatz(theta, nq))
        return float((psi.conj().T @ H @ psi).real)

    rng = np.random.default_rng(7)
    best = None
    for repeat in range(3):
        start = rng.uniform(-np.pi, np.pi, n_angles)
        result = minimize(energy, start, method="BFGS")
        if best is None or result.fun < best.fun:
            best = result
    return best.x, q_state(ansatz(best.x, nq))

# The same call also works on two qubits: q_optimise(q_ry, 1, H2(1.0), 2).
q_energies = []
for h in h_values:
    H = H6(h)
    theta_opt, psi_opt = q_optimise(q_my_ansatz, q_n_angles, H, 6)
    E = (psi_opt.conj().T @ H @ psi_opt).real
    q_energies.append(E / 6)

plt.figure(figsize=(6.5, 4))
plt.plot(h_values, exact_curve, label="Exact: six-qubit matrix")
plt.plot(h_values, mf_curve, "--", label="RY only: analytical minimum")
plt.plot(h_values, q_energies, "o--", label="My Qiskit circuit")
plt.xlabel(r"$h/J$")
plt.ylabel(r"$E/(6J)$")
plt.title("Qiskit: six qubits")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Check the distant-spin correlations of your Qiskit circuit at h=1 as well.
H = H6(1.0)
theta_opt, psi_opt = q_optimise(q_my_ansatz, q_n_angles, H, 6)
plt.figure(figsize=(6.5, 4))
plt.plot(r_values, z_correlations(psi_exact6), "o-", label="Exact")
plt.plot(r_values, z_correlations(psi_opt), "s--", label="My Qiskit circuit")
plt.xticks(r_values)
plt.xlabel("Separation r")
plt.ylabel(r"$\langle Z_0 Z_r\rangle$")
plt.title("Qiskit: six qubits, h/J = 1")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

*Based on the two supplied quantum-circuit tutorials. The instruction-list simulator is retained, with numerical gate functions instead of symbolic matrices. The energies use the lecture convention $-JZZ-hX$, rather than the older coupling prefactors. The two-qubit and mean-field formulas are derived above. The six-qubit reference is the exact eigenvalue of the same 64 × 64 matrix used for VQE.*